# Neural Network for pendulum stabilization

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

import gymnasium as gym # Environment with pole-cart example
from datetime import datetime
import time

# Gpu activation

In [ ]:
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print(f"Using device: MPS (Apple Silicon GPU - 14 cores)")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Using device: CUDA ({torch.cuda.get_device_name(0)})")
else:
    device = torch.device("cpu")
    print(f"Using device: CPU")

device = torch.device("cpu")
print(f"PyTorch version: {torch.__version__}")

# Model

In [ ]:
class PPO_CartPole_Model(nn.Module):

    def __init__(self,n):
        super().__init__()

        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 16)

        self.actor = nn.Linear(16,n)
        self.critic = nn.Linear(16,1)

    def forward(self,x):
        x = F.tanh(self.fc1(x))
        x = F.tanh(self.fc2(x))

        action_probs = F.softmax(self.actor(x),dim=-1)
        state_value = self.critic(x)

        return action_probs, state_value


# Optimizer and model initialization

In [ ]:
n = 16 # Num distribution bins
model = PPO_CartPole_Model(n)
model = model.to(device)
#model.load_state_dict(torch.load('InvertedPendulumPPO_V1_11-01-2026 10:13 PM.pt'))
optimizer = torch.optim.Adam(model.parameters(),lr = 3e-4)

# Create custom Distribution

In [ ]:
def get_action_value(idx,range_start,range_end,n):
    val_distributions = np.linspace(range_start,range_end,n)
    torque = torch.FloatTensor([val_distributions[idx]])
    return torque


# Storing all rewards

In [ ]:
all_rewards = []

# Training

In [ ]:
start_time = time.time()
epoch_time = time.time()

env = gym.make("InvertedPendulum-v5")

epochs = 10
K_epochs = 10
gamma = 0.99
clip_range = 0.2

coeff1 = 1
coeff2 = 0.5
coeff3 = 0.01

test = False

# Tracking lists
all_rewards = []
all_L_clip = []
all_L_VF = []
all_entropy = []
all_ratios = []  # Track how much policy is changing

for i in range(epochs):
    state,_ = env.reset()
    state = torch.FloatTensor(state).to(device)

    log_probs = []
    values = []
    rewards = []
    states = []
    actions = []
    done = False
    reward = 0

    # Collect one episode of experience
    while not done:
        states.append(state)
        action_probs, state_value = model(state)

        dist = torch.distributions.Categorical(action_probs)

        action_idx = dist.sample()
        action = get_action_value(action_idx.item(),-3,3,n)
        actions.append(action_idx)

        log_probs.append(dist.log_prob(action_idx))
        values.append(state_value)

        next_state, reward, terminated, truncated, _ = env.step([action.item()])


        done = terminated or truncated
        rewards.append(reward)
        state = torch.FloatTensor(next_state).to(device)

    ### Saving
    all_rewards.append(sum(rewards))
    log_probs = torch.stack(log_probs).detach().squeeze()
    actions = torch.stack(actions).squeeze()
    states = torch.stack(states).detach()
    print(actions)

    # Calculating advantage
    returns = []
    G = 0
    for reward in reversed(rewards):
        G = reward + gamma*G
        G = torch.FloatTensor([G], device=device)
        returns.insert(0,G)
    returns = torch.stack(returns)

    with torch.no_grad():
        _,old_values = model(states)

    advantage = returns - old_values
    advantage = advantage.squeeze()
    advantage = advantage.detach()


    for _ in range(K_epochs):
        action_probs, new_values = model(states)
        new_dist = torch.distributions.Categorical(action_probs)
        new_log_probs = new_dist.log_prob(actions)

        ratio = torch.exp(new_log_probs - log_probs)
        clipped_ratio = torch.clamp(ratio,1-clip_range,1+clip_range)

        L_clip = torch.min(ratio*advantage,clipped_ratio*advantage)
        L_vf = (new_values.squeeze() - returns.squeeze())**2
        entropy_bonus = new_dist.entropy()


        L_clip = L_clip * coeff1
        L_vf = L_vf * coeff2
        entropy_bonus = entropy_bonus * coeff3
        loss = -(L_clip - L_vf + entropy_bonus).mean()

        # Track metrics
        all_L_clip.append(L_clip.mean().item())
        all_L_VF.append(L_vf.mean().item())
        all_entropy.append(entropy_bonus.mean().item())
        all_ratios.append(ratio.mean().item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Print every 100 epochs
    if (i + 1) % 100 == 0:
        current_time = time.time()

        # Get last 100 epochs of data (each epoch has K_epochs entries)
        n_recent = 100 * K_epochs

        # Calculate averages
        avg_reward = sum(all_rewards[-100:]) / min(100, len(all_rewards[-100:]))
        avg_L_clip = sum(all_L_clip[-n_recent:]) / min(n_recent, len(all_L_clip[-n_recent:]))
        avg_L_vf = sum(all_L_VF[-n_recent:]) / min(n_recent, len(all_L_VF[-n_recent:]))
        avg_entropy = sum(all_entropy[-n_recent:]) / min(n_recent, len(all_entropy[-n_recent:]))
        avg_ratio = sum(all_ratios[-n_recent:]) / min(n_recent, len(all_ratios[-n_recent:]))

        # Min/max rewards in last 100
        recent_rewards = all_rewards[-100:]
        min_reward = min(recent_rewards)
        max_reward = max(recent_rewards)

        print(f"Epoch: {i+1}/{epochs} | Time: {current_time - epoch_time:.2f}s, "
              f"Avg: {avg_reward:.2f} | Min: {min_reward:.2f} | Max: {max_reward:.2f},"
              f" L_clip: {avg_L_clip:.6f} | L_vf: {avg_L_vf:.6f} | Entropy: {avg_entropy:.6f}")

        epoch_time = time.time()

current_time = time.time()
total_time = current_time - start_time
print(f"\n{'='*60}")
print(f"Training complete! Total time: {total_time/60:.2f} minutes")
print(f"Final avg reward (last 100): {sum(all_rewards[-100:])/100:.2f}")

# Histogram for rewards

In [ ]:
import matplotlib.pyplot as plt

plt.hist(all_rewards,bins=100)

# Visualize with Pygame

In [ ]:
# Visualize with Gymnasium

env_visual = gym.make('InvertedPendulum-v5', render_mode='human')
state, _ = env_visual.reset()

for _ in range(500):
    state_tensor = torch.FloatTensor(state)
    with torch.no_grad():
        action_probs, state_value = model(state_tensor)
        dist = torch.distributions.Categorical(action_probs)
        action_idx = dist.sample()
        action = get_action_value(action_idx.item(), -3, 3, n)

    # Ensure action is a numpy array with correct shape
    if isinstance(action, torch.Tensor):
        action = action.numpy()
    action = np.atleast_1d(np.array(action, dtype=np.float32))

    state, reward, terminated, truncated, _ = env_visual.step(action)
    if terminated or truncated:
        state, _ = env_visual.reset()

env_visual.close()

# Saving Model to visualize In Python Script

In [ ]:
#time_str = datetime.now().strftime("%d-%m-%Y %I:%M %p")
#model_name = 'Pendulum Models/InvertedPendulumPPO_V1_' + time_str + '.pt'
#torch.save(model.state_dict(), model_name)